In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import snapatac2 as snap
import matplotlib.pyplot as plt
import seaborn as sns
from plotnine import *
import pyranges as pr
import os
import warnings
import requests
from importlib import reload
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import scipy.cluster.hierarchy as hc
from scipy.spatial.distance import squareform
from umap.distances import hellinger

from tqdm import tqdm
tqdm.pandas()

import sys
sys.path.append('/scratch/eli')
sys.path.append('/data1/normantm/eli/software')
from perturbseq import *
from sparsepca import *
import multiome as mo

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42

In [2]:
gex = sc.read("/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_dg50_gex_raw.h5ad")
tgts = set([t.split("_")[0] for t in gex.obs.target_a.unique() if not pd.isna(t)])
tgts = [t.replace('NKX25', 'NKX2-5').replace('NKX31', 'NKX3-1') for t in tgts]
meanpop = gex.to_df().join(gex.obs.guide_target).groupby('guide_target').mean().drop("NTC", axis = 0)
degs = pd.read_csv("/data1/normantm/eli/T7/202511_RPE1_dg50/analysis/intermediate_files/251204_degs.csv")
meanpop_fit = meanpop.filter(items = degs.query("q < 0.01 and (z > 0.25 or z < -0.25)").gene.unique(), axis = 1)

gene_scores = pd.read_csv('/data1/normantm/multiome_collab/RPE1/amd_eurmeta_natgen_pops_scores.csv')
meanpop_fit = meanpop_fit.filter(items = gene_scores.HGNC.unique())

def zero_tgt(guide_target):
    
    a, b = guide_target.split("_") if "_" in guide_target else (guide_target, None)
    if a == 'NKX25':
        a = 'NKX2-5'
    if a == 'NKX31':
        a = 'NKX3-1'
    if b == 'NKX25':
        b = 'NKX2-5'
    if b == 'NKX31':
        b = 'NKX3-1'

    if a in meanpop_fit.columns:
        meanpop_fit.loc[meanpop_fit.index.str.contains(a.split("-")[0]), a] = 0.0
    if b in meanpop_fit.columns:
        meanpop_fit.loc[meanpop_fit.index.str.contains(b.split("-")[0]), b] = 0.0

for t in tgts:
    zero_tgt(t)

/tmp/ipykernel_1496468/686172767.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


In [3]:
sparsepca = NonNegativeSparsePCA(n_components=100, alpha = 1, method = 'cd', max_iter = 1000, n_jobs = 8, random_state = 42)

programs = sparsepca.fit_transform(meanpop_fit)
components_diff = pd.DataFrame(sparsepca.components_, columns=meanpop_fit.columns).reset_index().rename({'index':'program_id'}, axis = 1)
programs_diff = components_diff.melt(id_vars = 'program_id', var_name = 'feature', value_name = 'weight').query("weight != 0").sort_values(["program_id", "weight"], ascending = False)

In [ ]:
df_normalized_programs_diff = pd.DataFrame(programs, index = meanpop_fit.index)
df_normalized_programs_diff.loc['nonzero',:] = df_normalized_programs_diff.apply(lambda row: (row != 0).sum(), axis = 0)
df_normalized_programs_diff = df_normalized_programs_diff.T.query("nonzero > 0").T.drop("nonzero", axis = 0)

In [ ]:
# sort by weight within each program (descending)
programs_diff = pd.read_csv("intermediate_files/260414_gex_raw_100_prgm_features.csv")
df_sorted = programs_diff.sort_values(['program_id', 'weight'], ascending=[True, False])

# take top 10 per program
top10 = (
    df_sorted
    .groupby('program_id')
    .head(10)
)

# rank them 1–10 within each program
top10['rank'] = top10.groupby('program_id').cumcount() + 1

# pivot to wide format
result = (
    top10
    .pivot(index='program_id', columns='rank', values='feature')
)

mo.show(result)

In [ ]:
programs_diff.to_csv("intermediate_files/260414_gex_raw_100_prgm_features.csv")
df_normalized_programs_diff.to_csv("intermediate_files/260414_gex_raw_100_prgm_weights.csv")